# CoherenceBench-IN — Phase 1: Literature Review & Gap Validation
**Weeks 1–4 | ~20–25 hours**

### Goals
1. Build the **benchmark landscape table** (15+ entries) — confirms our gap is real
2. Run the **Semantic Scholar gap search** — find papers that show LLMs failing at coherence
3. Survey **Wikipedia corpus sizes** — confirm English data is abundant
4. Collect **5+ gap-evidence quotes** from 2023–2025 papers
5. Write the **draft introduction** (first 2 paragraphs)

### Phase 1 exit criteria checklist (fill in as you go)
- [ ] Benchmark landscape table complete (15+ entries)
- [ ] Confirmed: no benchmark tests all 3 coherence dimensions with injection
- [ ] TLDM analyzed — differentiation documented
- [ ] Wikipedia data survey shows English corpus is abundant  
- [ ] 5+ gap-evidence citations with exact quotes collected
- [ ] Draft introduction paragraph 1 + 2 written

In [ ]:
# ─── Imports ───────────────────────────────────────────────────────
import requests
import pandas as pd
import json
import time
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 50)

print("✅ Imports ready.")

## Part 1 — Benchmark Landscape Table
Search for existing long-context benchmarks using Semantic Scholar's free API, then build the landscape table that becomes Table 1 in your paper.

In [ ]:
# ─── Semantic Scholar Search Helper ───────────────────────────────
# Free API — no key needed. Rate limit: 1 req/sec.

S2_BASE = "https://api.semanticscholar.org/graph/v1"
S2_FIELDS = "title,authors,year,venue,externalIds,abstract,citationCount"

def s2_search(query: str, limit: int = 20) -> list[dict]:
    """Search Semantic Scholar and return a list of paper dicts."""
    url = f"{S2_BASE}/paper/search"
    params = {"query": query, "limit": limit, "fields": S2_FIELDS}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json().get("data", [])

def s2_paper(paper_id: str) -> dict:
    """Fetch a single paper by S2 ID, arXiv ID, or DOI."""
    url = f"{S2_BASE}/paper/{paper_id}"
    resp = requests.get(url, params={"fields": S2_FIELDS}, timeout=30)
    resp.raise_for_status()
    return resp.json()

def fmt_authors(authors: list) -> str:
    if not authors:
        return ""
    names = [a["name"].split()[-1] for a in authors[:3]]
    return ", ".join(names) + (" et al." if len(authors) > 3 else "")

print("✅ Semantic Scholar helper ready.")

In [ ]:
# ─── Fetch Known Benchmark Papers from Semantic Scholar ───────────
# arXiv IDs for the 6 core papers to read (from plan.md)
CORE_PAPERS = {
    "RULER":          "arXiv:2404.06654",
    "Lost-in-Middle": "arXiv:2307.03172",
    "LongBench-v2":   "arXiv:2412.15204",
    "HELMET":         "arXiv:2410.02694",
    "InfiniteBench":  "arXiv:2402.13718",
    # TLDM (Hamilton 2025) — search by title if no arXiv ID yet
}

fetched = []
for name, pid in CORE_PAPERS.items():
    try:
        p = s2_paper(pid)
        fetched.append({
            "name": name,
            "title": p.get("title", ""),
            "authors": fmt_authors(p.get("authors", [])),
            "year": p.get("year", ""),
            "venue": p.get("venue", "arXiv"),
            "citations": p.get("citationCount", 0),
            "abstract_snippet": (p.get("abstract") or "")[:200] + "...",
        })
        time.sleep(1)  # respect rate limit
    except Exception as e:
        print(f"  ⚠️  Could not fetch {name}: {e}")

df_core = pd.DataFrame(fetched)
display(df_core[["name", "authors", "year", "venue", "citations"]])

# Also search for TLDM
print("\nSearching for TLDM...")
tldm_results = s2_search("Too Long Didn't Model long context novel comprehension", limit=5)
for r in tldm_results:
    print(f"  {r['year']} | {r['title'][:80]} | {r['citationCount']} cites")

In [ ]:
# ─── Benchmark Landscape Table (pre-populated + fill as you read) ──
# Update "coherence", "injection", "languages" as you read each paper.
# Columns:
#   coherence: None | Partial | Full
#   injection: does it inject controlled corruptions?
#   max_ctx_k: max context length in thousands of tokens

LANDSCAPE = [
    # ── Known benchmarks (fill details as you read) ──────────────
    {"name": "RULER",           "year": 2024, "focus": "Retrieval (NIAH variants)",         "max_ctx_k": 128,  "languages": "EN",       "coherence": "None",    "injection": False, "key_limit": "Retrieval only, synthetic distractors"},
    {"name": "LongBench v2",    "year": 2024, "focus": "Multi-task QA, summ, code",         "max_ctx_k": 35,   "languages": "EN/ZH",    "coherence": "None",    "injection": False, "key_limit": "No coherence dimension"},
    {"name": "InfiniteBench",   "year": 2024, "focus": "Retrieval + QA",                    "max_ctx_k": 200,  "languages": "EN/ZH",    "coherence": "None",    "injection": False, "key_limit": "Retrieval, no coherence"},
    {"name": "HELMET",          "year": 2024, "focus": "Holistic: recall, CITE, QA, summ",  "max_ctx_k": 128,  "languages": "EN",       "coherence": "Partial", "injection": False, "key_limit": "Summ coherence surface-only, no injection"},
    {"name": "Lost-in-Middle",  "year": 2024, "focus": "Positional bias in retrieval",      "max_ctx_k": 20,   "languages": "EN",       "coherence": "None",    "injection": False, "key_limit": "Analysis paper, not a reusable benchmark"},
    {"name": "TLDM",            "year": 2025, "focus": "Novel comprehension",               "max_ctx_k": 100,  "languages": "EN",       "coherence": "Partial", "injection": False, "key_limit": "Novels only, no injection, no Indian langs"},
    {"name": "L-Eval",          "year": 2023, "focus": "Long-doc QA, summ",                "max_ctx_k": 60,   "languages": "EN",       "coherence": "None",    "injection": False, "key_limit": "No coherence tasks"},
    {"name": "SCROLLS",        "year": 2022, "focus": "Long-doc summarization, QA",        "max_ctx_k": 65,   "languages": "EN",       "coherence": "None",    "injection": False, "key_limit": "Pre-LLM era, no coherence"},
    {"name": "ZeroScrolls",    "year": 2023, "focus": "Zero-shot long QA, summ",           "max_ctx_k": 65,   "languages": "EN",       "coherence": "None",    "injection": False, "key_limit": "No coherence tasks"},
    {"name": "LooGLE",          "year": 2024, "focus": "Long-doc QA (interdep. questions)", "max_ctx_k": 24,   "languages": "EN",       "coherence": "Partial", "injection": False, "key_limit": "Time-series QA, no controlled injection"},
    # ── ADD MORE as you read ── (copy a row and fill in the values)
    # {"name": "",   "year": 2024, "focus": "",  "max_ctx_k": 0, "languages": "EN", "coherence": "None", "injection": False, "key_limit": ""},

    # ── Our contribution (for comparison) ────────────────────────
    {"name": "CoherenceBench-IN (OURS)", "year": 2026, "focus": "Entity + Temporal + Causal coherence", "max_ctx_k": 64, "languages": "EN/HI/TA", "coherence": "Full", "injection": True, "key_limit": "—"},
]

df_landscape = pd.DataFrame(LANDSCAPE)

# Color-code coherence coverage
def highlight_coherence(val):
    colors = {"None": "background-color: #ffcccc", "Partial": "background-color: #fff3cc", "Full": "background-color: #ccffcc"}
    return colors.get(val, "")

print("=" * 80)
print("BENCHMARK LANDSCAPE TABLE  (Table 1 of paper)")
print("=" * 80)
display(df_landscape.style.applymap(highlight_coherence, subset=["coherence"]))

# Summary
print(f"\nTotal benchmarks surveyed: {len(df_landscape)}")
print(f"  Coherence = None:    {(df_landscape['coherence'] == 'None').sum()}")
print(f"  Coherence = Partial: {(df_landscape['coherence'] == 'Partial').sum()}")
print(f"  Coherence = Full:    {(df_landscape['coherence'] == 'Full').sum()}  ← only us")

In [ ]:
# ─── Gap Analysis Matrix ───────────────────────────────────────────
# Shows which specific coherence capabilities each benchmark covers.
# ✅ = covered  ⚠️ = partial  ❌ = not covered

GAP_MATRIX = [
    {"Benchmark":           "RULER",               "Entity Consistency": "❌", "Temporal Coherence": "❌", "Causal Chains": "❌", "Injection Methodology": "❌", "Indian Languages": "❌"},
    {"Benchmark":           "LongBench v2",        "Entity Consistency": "❌", "Temporal Coherence": "❌", "Causal Chains": "❌", "Injection Methodology": "❌", "Indian Languages": "❌"},
    {"Benchmark":           "InfiniteBench",       "Entity Consistency": "❌", "Temporal Coherence": "❌", "Causal Chains": "❌", "Injection Methodology": "❌", "Indian Languages": "❌"},
    {"Benchmark":           "HELMET",              "Entity Consistency": "❌", "Temporal Coherence": "⚠️", "Causal Chains": "❌", "Injection Methodology": "❌", "Indian Languages": "❌"},
    {"Benchmark":           "TLDM",                "Entity Consistency": "⚠️", "Temporal Coherence": "⚠️", "Causal Chains": "❌", "Injection Methodology": "❌", "Indian Languages": "❌"},
    {"Benchmark":           "L-Eval",              "Entity Consistency": "❌", "Temporal Coherence": "❌", "Causal Chains": "❌", "Injection Methodology": "❌", "Indian Languages": "❌"},
    {"Benchmark":           "LooGLE",              "Entity Consistency": "⚠️", "Temporal Coherence": "⚠️", "Causal Chains": "❌", "Injection Methodology": "❌", "Indian Languages": "❌"},
    # ── Add more as you read ──
    {"Benchmark":           "CoherenceBench-IN ✨", "Entity Consistency": "✅", "Temporal Coherence": "✅", "Causal Chains": "✅", "Injection Methodology": "✅", "Indian Languages": "✅"},
]

df_gaps = pd.DataFrame(GAP_MATRIX).set_index("Benchmark")

print("=" * 70)
print("GAP ANALYSIS MATRIX  (becomes part of Related Work section)")
print("=" * 70)
display(df_gaps)

# Our unique combination
cols = ["Entity Consistency","Temporal Coherence","Causal Chains","Injection Methodology","Indian Languages"]
fully_covered = df_gaps[df_gaps[cols].eq("✅").all(axis=1)]
print(f"\nBenchmarks covering ALL 5 capabilities: {len(fully_covered)}")
print("→ Only CoherenceBench-IN covers the full combination. Gap is confirmed ✅")

---
## Part 2 — Wikipedia Corpus Survey
Verify that English Wikipedia has enough long articles for our corpus. Indian language survey runs here too but is low priority — English first.

In [ ]:
# ─── Wikipedia Corpus Survey ───────────────────────────────────────
# Stream each language's Wikipedia and sample 500 articles.
# Measure token length distribution to confirm we have enough long articles.

from datasets import load_dataset
import tiktoken
import numpy as np
import matplotlib.pyplot as plt

enc = tiktoken.get_encoding("cl100k_base")

# LANGUAGE PRIORITY: English first. Hindi/Tamil are last.
# Set survey_indian=True only after English pipeline is complete.
SURVEY_LANGS = [
    ("en", "20231101.en", "English",  "PRIORITY 1"),
    # ("hi", "20231101.hi", "Hindi",  "PRIORITY 3 — run later"),
    # ("ta", "20231101.ta", "Tamil",  "PRIORITY 3 — run later"),
]

MIN_TOKENS = 4_000   # our short-context threshold
SAMPLE_SIZE = 500    # articles to sample per language

results = {}

for lang_code, config, lang_name, priority in SURVEY_LANGS:
    print(f"\n{'='*60}")
    print(f"Language: {lang_name} ({priority})")
    print(f"Config:   wikimedia/wikipedia {config}")
    print(f"Sampling {SAMPLE_SIZE} articles...")

    ds = load_dataset("wikimedia/wikipedia", config, split="train", streaming=True)

    lengths = []
    for i, article in enumerate(ds):
        if i >= SAMPLE_SIZE:
            break
        n = len(enc.encode(article["text"]))
        lengths.append(n)

    lengths = np.array(lengths)
    long_mask = lengths >= MIN_TOKENS

    results[lang_code] = {
        "lang": lang_name,
        "sampled": SAMPLE_SIZE,
        "mean_tokens": int(lengths.mean()),
        "median_tokens": int(np.median(lengths)),
        "pct_above_4k": f"{long_mask.mean()*100:.1f}%",
        "pct_above_8k": f"{(lengths >= 8_000).mean()*100:.1f}%",
        "pct_above_32k": f"{(lengths >= 32_000).mean()*100:.1f}%",
        "lengths": lengths,
    }

    r = results[lang_code]
    print(f"  Mean tokens:         {r['mean_tokens']:,}")
    print(f"  Median tokens:       {r['median_tokens']:,}")
    print(f"  Articles ≥ 4K tok:  {r['pct_above_4k']}")
    print(f"  Articles ≥ 8K tok:  {r['pct_above_8k']}")
    print(f"  Articles ≥ 32K tok: {r['pct_above_32k']}")

# Plot length distribution
fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 4))
if len(results) == 1:
    axes = [axes]
for ax, (lang_code, r) in zip(axes, results.items()):
    ax.hist(np.clip(r["lengths"], 0, 50_000), bins=50, color="steelblue", edgecolor="white")
    ax.axvline(4_000, color="red", linestyle="--", label="4K threshold")
    ax.axvline(32_000, color="orange", linestyle="--", label="32K threshold")
    ax.set_title(f"{r['lang']} Wikipedia — token length distribution\n(n={SAMPLE_SIZE}, clipped at 50K)")
    ax.set_xlabel("Tokens")
    ax.set_ylabel("# Articles")
    ax.legend()
plt.tight_layout()
plt.savefig("../results/wikipedia_token_distribution.png", dpi=150)
plt.show()
print("\n✅ Saved to results/wikipedia_token_distribution.png")

---
## Part 3 — Gap Evidence Search
Find 5–8 papers (2023–2025) that either *show* LLMs failing at coherence, *or* explicitly state coherence evaluation is a gap. These quotes go directly into your Introduction as motivation.

In [ ]:
# ─── Gap Evidence: Semantic Scholar Search ─────────────────────────
# Run multiple targeted queries to find papers showing LLMs failing at coherence.

GAP_QUERIES = [
    "LLM discourse coherence failure long context",
    "large language models inconsistency entity long document",
    "temporal reasoning consistency LLM evaluation",
    "causal reasoning failure large language model",
    "long context coherence evaluation benchmark gap",
    "LLM hallucination entity inconsistency long text",
]

all_hits = []
seen_ids = set()

for query in GAP_QUERIES:
    try:
        hits = s2_search(query, limit=10)
        for h in hits:
            pid = h.get("paperId", "")
            if pid not in seen_ids and h.get("year", 0) >= 2022:
                seen_ids.add(pid)
                all_hits.append({
                    "title":    h.get("title", ""),
                    "authors":  fmt_authors(h.get("authors", [])),
                    "year":     h.get("year", ""),
                    "venue":    h.get("venue", ""),
                    "citations":h.get("citationCount", 0),
                    "abstract": (h.get("abstract") or "")[:300],
                    "query":    query,
                })
        time.sleep(1)
    except Exception as e:
        print(f"  ⚠️  Query failed: {query} → {e}")

df_evidence = pd.DataFrame(all_hits).sort_values("citations", ascending=False)
print(f"Found {len(df_evidence)} unique papers (2022+) across {len(GAP_QUERIES)} queries")
print()
display(df_evidence[["title","authors","year","venue","citations"]].head(30))

In [ ]:
# ─── Gap Evidence Dossier (fill this as you read) ─────────────────
# For each paper: paste the exact quote that shows the gap, and explain why it
# supports CoherenceBench-IN. These become inline citations in the Introduction.

GAP_EVIDENCE = [
    # ── Pre-filled (from prior literature search) ─────────────────────────
    {
        "citation":  "Hsieh et al. (2024) — RULER",
        "finding":   "Most models claiming 32K+ context support degrade significantly beyond their effective context size.",
        "quote":     "Performance of all models decreases as sequence length increases... Several long-context models fail dramatically even at context sizes they claim to support.",
        "why_it_helps": "Shows that even simple retrieval degrades with length — coherence (harder) will degrade more. Motivates our distance analysis.",
        "verified":  True,
    },
    # ── Fill these as you read ─────────────────────────────────────────────
    {
        "citation":  "Liu et al. (2024) — Lost in the Middle",
        "finding":   "[TO FILL after reading]",
        "quote":     "[PASTE EXACT QUOTE HERE]",
        "why_it_helps": "[TO FILL]",
        "verified":  False,
    },
    {
        "citation":  "[AUTHOR et al., YEAR] — [PAPER NAME]",
        "finding":   "[TO FILL]",
        "quote":     "[PASTE EXACT QUOTE HERE]",
        "why_it_helps": "[TO FILL]",
        "verified":  False,
    },
    {
        "citation":  "[AUTHOR et al., YEAR] — [PAPER NAME]",
        "finding":   "[TO FILL]",
        "quote":     "[PASTE EXACT QUOTE HERE]",
        "why_it_helps": "[TO FILL]",
        "verified":  False,
    },
    {
        "citation":  "[AUTHOR et al., YEAR] — [PAPER NAME]",
        "finding":   "[TO FILL]",
        "quote":     "[PASTE EXACT QUOTE HERE]",
        "why_it_helps": "[TO FILL]",
        "verified":  False,
    },
]

# Display dossier
verified = [e for e in GAP_EVIDENCE if e["verified"]]
todo     = [e for e in GAP_EVIDENCE if not e["verified"]]

print(f"Gap evidence dossier: {len(verified)} verified, {len(todo)} to fill")
print(f"Target: 5 verified entries\n")
for i, e in enumerate(GAP_EVIDENCE, 1):
    status = "✅" if e["verified"] else "⬜"
    print(f"{status} [{i}] {e['citation']}")
    print(f"     Finding: {e['finding'][:100]}")
    if e["verified"]:
        print(f'     Quote:   "{e["quote"][:120]}"')
    print()

---
## Part 4 — Draft Introduction (Week 4)
Fill in the `[BRACKET]` placeholders below as you read. This becomes the first ~400 words of your paper. Keep it factual — every claim needs a citation from your dossier above.

In [ ]:
# ─── Draft Introduction — fill [BRACKETS] as you read ─────────────
# After you complete the gap evidence dossier, edit the strings below
# and run this cell. The output is your Overleaf Introduction §1 draft.

INTRO_P1 = """
Large Language Models (LLMs) have rapidly expanded their context windows from
4,096 tokens in 2023 to over one million tokens in 2025–2026, enabling
applications such as [EXAMPLE APPLICATION 1], [EXAMPLE APPLICATION 2], and
end-to-end processing of [EXAMPLE DOCUMENT TYPE].
Yet the dominant paradigm for evaluating these extended context capabilities
remains retrieval-centric: benchmarks such as RULER [CITE Hsieh 2024],
LongBench v2 [CITE Bai 2024], and InfiniteBench [CITE Zhang 2024] measure
whether a model can *locate* a specific fact buried within a long context
("needle-in-a-haystack"), but not whether the model *understands the structure*
of the document it has processed.
"""

INTRO_P2 = """
We identify a critical and underexplored gap: no existing benchmark evaluates
**discourse coherence** in LLMs — that is, whether a model maintains
consistent entities, valid timelines, and intact causal chains across long-form
text. This gap is consequential. [AUTHOR et al., YEAR] showed that [GAP FINDING 1 WITH BRIEF QUOTE].
[AUTHOR et al., YEAR] further demonstrated that [GAP FINDING 2].
A model that achieves near-perfect recall on RULER [CITE] may silently accept
a document that contradicts an entity's occupation at token 15,000, inverts
the chronological order of events, or breaks a causal relationship established
thousands of tokens earlier. These failures are silent at inference time —
they produce no error signal, only subtly wrong outputs that are difficult to
catch without explicit coherence evaluation.
"""

INTRO_P3 = """
We present **CoherenceBench-IN**, the first benchmark specifically designed to
evaluate discourse coherence in LLMs. Our benchmark covers three formally
defined dimensions: (1) *entity consistency* — whether named entities'
attributes remain non-contradictory across long spans; (2) *temporal coherence*
— whether event sequences, dates, and durations remain logically valid; and
(3) *causal chain integrity* — whether cause-effect relationships established
early in a document are preserved throughout. CoherenceBench-IN employs a novel
**controlled incoherence injection** methodology: clean long-form texts sourced
from Wikipedia and Project Gutenberg are programmatically corrupted at
configurable token distances, enabling systematic measurement of how model
sensitivity to coherence violations degrades as the corruption moves farther
from the evaluation question — a distance effect motivated by the positional
bias findings of [CITE Liu 2024].
"""

INTRO_CONTRIBUTIONS = """
In this paper, we make the following contributions:
(1) **CoherenceBench-IN**, a benchmark of ~900 instances evaluating
    discourse coherence across three dimensions and [N] languages;
(2) a **controlled incoherence injection methodology** that generates
    verifiable coherence violations at configurable token distances;
(3) a systematic evaluation of [N] LLMs revealing that coherence performance
    degrades [X]% more than retrieval performance in long contexts,
    with the degradation scaling with corruption distance [{CITE distance analysis}];
(4) the first coherence measurements for Indian languages (Hindi and [Tamil/Telugu]).
"""

full_draft = INTRO_P1 + INTRO_P2 + INTRO_P3 + INTRO_CONTRIBUTIONS

display(Markdown("### DRAFT INTRODUCTION\n---"))
display(Markdown(full_draft))

# Save to file
with open("../paper/draft_introduction.md", "w") as f:
    f.write("# Draft Introduction\n\n")
    f.write("*Fill [BRACKETS] with citations from gap evidence dossier.*\n\n")
    f.write(full_draft)
print("\n✅ Saved to paper/draft_introduction.md")

In [ ]:
# ─── Export All Tables to Markdown & CSV ──────────────────────────
# Run at end of each session to save progress.
import os
os.makedirs("../paper/tables", exist_ok=True)
os.makedirs("../references", exist_ok=True)

# 1. Benchmark landscape
df_landscape.to_csv("../references/benchmark_landscape.csv", index=False)
with open("../paper/tables/benchmark_landscape.md", "w") as f:
    f.write("# Benchmark Landscape Table\n\n")
    f.write(df_landscape.to_markdown(index=False))
    f.write("\n\n*Table 1. Existing long-context benchmarks and their coherence coverage.*\n")

# 2. Gap analysis matrix
df_gaps.to_csv("../references/gap_analysis.csv")
with open("../paper/tables/gap_analysis.md", "w") as f:
    f.write("# Gap Analysis Matrix\n\n")
    f.write(df_gaps.to_markdown())
    f.write("\n\n*✅ = covered  ⚠️ = partial  ❌ = not covered*\n")

# 3. Gap evidence dossier
import json
with open("../references/gap_evidence.json", "w") as f:
    json.dump(GAP_EVIDENCE, f, indent=2)

print("✅ Exported:")
print("   paper/tables/benchmark_landscape.md")
print("   paper/tables/gap_analysis.md")
print("   paper/draft_introduction.md")
print("   references/benchmark_landscape.csv")
print("   references/gap_analysis.csv")
print("   references/gap_evidence.json")

---
## Phase 1 Checklist — update as you complete each item

| # | Task | Status | Notes |
|---|------|--------|-------|
| 1 | Read RULER — write 1-paragraph summary | ⬜ | See `references/paper_notes.md` |
| 2 | Read Lost in the Middle — fill gap evidence entry | ⬜ | |
| 3 | Read LongBench v2 — update landscape table | ⬜ | |
| 4 | Read TLDM — write differentiation notes | ⬜ | Closest competitor |
| 5 | Read HELMET — update landscape table | ⬜ | |
| 6 | Read InfiniteBench — update landscape table | ⬜ | |
| 7 | Add 4+ more benchmarks to landscape table | ⬜ | Target: 15+ total |
| 8 | Run Wikipedia corpus survey (English) | ⬜ | Cell above |
| 9 | Read Centering Theory (Grosz et al., 1995) | ⬜ | Week 3 |
| 10 | Read Entity Grid (Barzilay & Lapata, 2008) | ⬜ | Week 3 |
| 11 | Collect 5 verified gap evidence entries | ⬜ | Fill dossier above |
| 12 | Fill draft introduction [BRACKETS] | ⬜ | Run intro cell |
| 13 | Export all tables | ⬜ | Run export cell |

**→ Next notebook:** `02_corpus_pipeline.ipynb` (Phase 2)